# k6 — Đánh giá 9 nhánh: pool + phiếu chấm mù → nhập nhãn → bảng điểm (CPU)

**Cài đặt:** Accelerator = **None (CPU)**, Internet = On.
**Add Input:** `mir2-code`, `mir2-data`, `mir2-chunks`, `mir2-runs` (output k4), output k5 (các run đã rerank, nếu có);
với `MODE = 'score'` / `'extend'` thêm dataset **`mir2-labels`** (thư mục `judge/` đã điền nhãn). Quy trình: docs/EVAL_PLAN.md.

- `MODE = 'pool'`: chẩn đoán không nhãn (kèm thay đổi do rerank) → chọn `N_QUERIES` câu phân tầng → pool + phiếu ẩn
  nguồn: `sheet_pilot_<người>.csv` (`PILOT` câu, MỌI người cùng chấm) và `sheet_pool_<người>.csv` (phần còn lại, chia
  nhau, 20% câu chấm chéo). Tải cả thư mục `eval/judge/` về, điền cột `grade (0/1/2)` (Google Sheets → *Download CSV*;
  Excel dễ hỏng dấu tiếng Việt), tạo dataset `mir2-labels` từ cả thư mục đó (giữ `key.parquet`, `queries.csv`).
  Chấm pilot trước → `MODE = 'score'` xem kappa pilot → họp, sửa hướng dẫn → mới chấm phiếu pool.
- `MODE = 'extend'`: thêm vào pool các mục CHƯA chấm của run mới (`EXTEND`, vd `'rrf9_w:30'`; run gộp có trọng số
  tạo ngay trong notebook qua `FUSE`) → phiếu bổ sung. Giữ nguyên `FUSE` ở lần `MODE = 'score'` sau đó.
- `MODE = 'score'`: nhập nhãn (kappa, bất đồng) → `eval/score.md`, `eval/marginal.md`, `eval/table_k*.md`.

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k6-v1 (15/10/2026)'
MODE = 'pool'                      # 'pool' | 'extend' | 'score'
N_QUERIES, PILOT = 60, 6
LABELERS = 'A,B,C'
FUSED = 'rrf9:30,rrf9_colbert:30,rrf9_cross:30'   # run gộp/rerank đưa vào pool (top-30)
EXTEND = ''                        # MODE='extend': 'run:độ sâu,…'
ADJUDICATED = ''                   # file phân xử (item,grade) trong mir2-labels, nếu có
FUSE = {}   # run gộp thử thêm, vd {'rrf9_w': 'vi_dense,vi_lsr,vi_bm25,en_dense,en_lsr,en_bm25,zh_dense,zh_lsr,zh_bm25*0.5'}
KS = '10,20,50,100,185,200'
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True), key=len)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=('chunks', 'qtrans'))

In [ ]:
# (2) Gom mọi run (k4 + k5) vào /kaggle/working/runs; chép thư mục nhãn (nếu có) vào eval/judge
OUT = f'{K.work}/runs'
os.makedirs(OUT, exist_ok=True)
for d in locate_all('runs'):
    for f in glob.glob(f'{d}/runs/*.parquet') + glob.glob(f'{d}/runs/*.json'):
        if not os.path.exists(f'{OUT}/' + os.path.basename(f)):
            shutil.copy(f, OUT)
K.set({'retrieve.out_dir': OUT})
print('run:', sorted(os.path.basename(f)[:-8] for f in glob.glob(f'{OUT}/*.parquet')))
JD = f'{K.work}/eval/judge'
lab = locate_all('judge/key.parquet')
if MODE != 'pool':
    assert lab, 'Chưa Add Input dataset nhãn (thư mục judge/ có key.parquet)'
    if len(lab) > 1:
        print('[CẢNH BÁO] nhiều thư mục judge/ trong input, dùng', lab[0], '— chỉ nên thêm bản mir2-labels mới nhất')
    shutil.copytree(lab[0] / 'judge', JD, dirs_exist_ok=True)
    print('nhãn từ', lab[0] / 'judge', ':', sorted(os.listdir(JD)))

In [ ]:
# (3) Chạy theo MODE (trước đó tạo các run gộp thử trong FUSE — rẻ, chạy lại mỗi lần)
for name, spec in FUSE.items():
    sh(f'python -m mirret.fuse --config {K.cfg} --runs {spec} --name {name}', 'fuse')
have = {os.path.basename(f)[:-8] for f in glob.glob(f'{OUT}/*.parquet')}
if MODE == 'pool':
    rr = ','.join(n for n in ('rrf9_colbert', 'rrf9_cross') if n in have)
    sh(f'python -m mirret.diag --config {K.cfg}' + (f' --reranked {rr}' if rr else ''), 'diag')
    sh(f'python -m mirret.judge select --config {K.cfg} --n {N_QUERIES} --pilot {PILOT}', 'select')
    fused = ','.join(x for x in FUSED.split(',') if x.split(':')[0] in have)
    sh(f'python -m mirret.judge pool --config {K.cfg} --fused {fused} --labelers {LABELERS}', 'pool')
elif MODE == 'extend':
    sh(f'python -m mirret.judge extend --config {K.cfg} --runs {EXTEND} --labelers {LABELERS}', 'extend')
else:
    adj = f' --adjudicated {lab[0] / "judge" / ADJUDICATED}' if ADJUDICATED else ''
    sh(f'python -m mirret.judge ingest --config {K.cfg}{adj}', 'ingest')
    allruns = ','.join(sorted(have))                  # gồm cả run từng mức (@parents/@children) và run rerank
    sh(f'python -m mirret.evaluate score --config {K.cfg} --ks {KS} --runs {allruns}', 'score')
    sh(f'python -m mirret.evaluate marginal --config {K.cfg} --k 50', 'marginal')
    for k in (20, 50, 200):
        sh(f'python -m mirret.evaluate table --config {K.cfg} --ks {k}', f'table_{k}')
shutil.rmtree(OUT)                                     # run đã có trong dataset input, không chép lại vào output
report('k6')